In [ ]:
from pathlib import Path

import torch
import numpy as np
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import anndata

In [ ]:
data_root = Path("./hest1k_datasets/PRAD")
processed_data_path = data_root / "processed_data"
st_path = data_root / "st"

slide_out = "MEND145"

# load training slide IDs
slidename_lst = list(np.genfromtxt(processed_data_path / "all_slide_lst.txt", dtype=str))
slidename_lst.remove(slide_out)
print(slide_out, " is held out for testing ", (slide_out not in slidename_lst))

# load selected gene list
selected_genes = list(np.genfromtxt(processed_data_path / "selected_gene_list_1k.txt", dtype=str))
print("Selected gene list loaded. len of selected genes: ", len(selected_genes))

# load test slide
test_adata = anndata.read_h5ad(st_path / f"{slide_out}.h5ad")
test_count_mtx_df = pd.DataFrame(test_adata.X.toarray(), columns=test_adata.var_names, index=test_adata.obs_names).loc[:, selected_genes]
# transform count
test_count_mtx_selected_genes = np.log2(test_count_mtx_df + 1).copy()
print("Test count mtx shape: ", test_count_mtx_selected_genes.shape)

In [ ]:
# load generated samples for test slide
sample_path = Path("./results/PRAD/runs/000/samples/generated_0015000_cond20_n5.pt")
pred = torch.load(sample_path, weights_only=False)
print("Generated samples shape: ", pred.shape)

In [ ]:
num_rep = 20       # number of samples generated for one image patch
num_selected = 20  # take average of (a subset / all) samples and use it as prediction
# num_rep = 2       # number of samples generated for one image patch
# num_selected = 2  # take average of (a subset / all) samples and use it as prediction
random_selected_index = np.random.choice(np.arange(num_rep), num_selected, replace=False)

pred_avg = torch.zeros(size=(test_count_mtx_selected_genes.shape[0], 
                             test_count_mtx_selected_genes.shape[1]))
for i in range(test_count_mtx_selected_genes.shape[0]):
    pred_avg[i] = torch.mean(pred[i*num_rep + random_selected_index, :], dim=0)
pred_avg = pred_avg.cpu().detach().numpy()


In [ ]:
# calculate correlation

all_corr = []
gene_corr_dict = {}
for i in range(test_count_mtx_selected_genes.shape[1]):
    gene_name = selected_genes[i]
    x = test_count_mtx_selected_genes.iloc[:, i].values
    y = pred_avg[:, i]
    cor = np.corrcoef(x, y)[0][1]
    all_corr.append(cor)
    gene_corr_dict[gene_name] = cor

plt.hist(all_corr, bins=50)

Evaluation metrics

In [ ]:
# evaluation metrics
# PCC
print("PCC-10: {:.4f}".format(np.mean(sorted(all_corr)[::-1][:10])))
print("PCC-50: {:.4f}".format(np.mean(sorted(all_corr)[::-1][:50])))
print("PCC-200: {:.4f}".format(np.mean(sorted(all_corr)[::-1][:200])))
# MSE, MAE
print("MAE: {:.4f}".format(np.mean(np.abs(test_count_mtx_selected_genes.values - pred_avg))))
print("MSE: {:.4f}".format(np.mean((test_count_mtx_selected_genes.values - pred_avg)**2)))
# RVD
pred_var = np.var(pred_avg, axis=0)
gt_var = np.var(test_count_mtx_selected_genes.values, axis=0)
print("RVD: {:.4f}".format(np.mean((pred_var - gt_var)**2 / gt_var**2)))

In [ ]:
# gene variation curve
fig, axs = plt.subplots(2, 2, figsize=(6, 6))

pred_mean = np.mean(pred_avg, axis=0)
pred_mean = pred_mean / np.sum(pred_mean)
gt_mean = np.mean(test_count_mtx_selected_genes, axis=0)
gt_mean = gt_mean / np.sum(gt_mean)
gt_mean_sorted = np.sort(gt_mean)
pred_mean_sorted = pred_mean[np.argsort(gt_mean)]
axs[0, 0].plot(np.arange(len(gt_mean_sorted)), gt_mean_sorted, label="Ground Truth", c="b")
axs[0, 0].scatter(np.arange(len(pred_mean_sorted)), pred_mean_sorted, s=5, label="Predicted", c="orange")
axs[0, 0].set_title("Normalized Mean")
axs[0, 0].set_xlabel("gene index ordered by mean")
axs[0, 0].set_ylabel("normalized mean")
# axs[0, 0].set_ylim()

pred_mean = np.mean(pred_avg, axis=0)
gt_mean = np.mean(test_count_mtx_selected_genes, axis=0)
gt_mean_sorted = np.sort(gt_mean)
pred_mean_sorted = pred_mean[np.argsort(gt_mean)]
axs[1, 0].plot(np.arange(len(gt_mean_sorted)), gt_mean_sorted, label="Ground Truth", c="b")
axs[1, 0].scatter(np.arange(len(pred_mean_sorted)), pred_mean_sorted, s=5, label="Predicted", c="orange")
axs[1, 0].set_title("Absolute Mean")
axs[1, 0].set_xlabel("gene index ordered by mean")
axs[1, 0].set_ylabel("absolute mean")
# axs[1, 0].set_ylim()


pred_var = np.var(pred_avg, axis=0)
pred_var = pred_var / np.sum(pred_var)
gt_var = np.var(test_count_mtx_selected_genes, axis=0)
gt_var = gt_var / np.sum(gt_var)
gt_var_sorted = np.sort(gt_var)
pred_var_sorted = pred_var[np.argsort(gt_var)]
axs[0, 1].plot(np.arange(len(gt_var_sorted)), gt_var_sorted, label="Ground Truth", c="b")
axs[0, 1].scatter(np.arange(len(pred_var_sorted)), pred_var_sorted, s=5, label="Predicted", c="orange")
axs[0, 1].set_title("Normalized Variance")
axs[0, 1].set_xlabel("gene index ordered by var")
axs[0, 1].set_ylabel("normalized variance")
# axs[0, 1].set_ylim()



pred_var = np.var(pred_avg, axis=0)
gt_var = np.var(test_count_mtx_selected_genes, axis=0)
gt_var_sorted = np.sort(gt_var)
pred_var_sorted = pred_var[np.argsort(gt_var)]
axs[1, 1].plot(np.arange(len(gt_var_sorted)), gt_var_sorted, label="Ground Truth", c="b")
axs[1, 1].scatter(np.arange(len(pred_var_sorted)), pred_var_sorted, s=5, label="Predicted", c="orange")
axs[1, 1].set_title("Absolute Variance")
axs[1, 1].set_xlabel("gene index ordered by var")
axs[1, 1].set_ylabel("absolute variance")
# axs[1, 1].set_ylim()

plt.tight_layout()
plt.show()

In [ ]:
# Sort genes by their correlation coefficient in descending order
sorted_gene_corr = sorted(gene_corr_dict.items(), key=lambda x: x[1], reverse=True)
top_10_df = pd.DataFrame(
    {
        'Rank': range(1, 11),
        'Gene': [gene for gene, corr in sorted_gene_corr[:10]],
        'Correlation': [f"{corr:.4f}" for gene, corr in sorted_gene_corr[:10]]
    }
)
print("=" * 40)
print("Top 10 genes by correlation coefficient")
print("=" * 40)
print(top_10_df.to_string(index=False))
print("=" * 40)


In [ ]:

Image.MAX_IMAGE_PIXELS = None

image_path = data_root / "wsis" / f"{slide_out}.tif"
img_raw = Image.open(image_path)  # Adjust the suffix for your WSI format.

x = test_adata.obsm["spatial"][:, 0]
y = test_adata.obsm["spatial"][:, 1]
for gene in ["KLK2"]: # fill in gene names
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))
    fig.suptitle(gene)
    gene_idx = np.where(np.array(selected_genes) == gene)[0][0]

    axs[0].imshow(img_raw)
    color_gt = test_adata[:, test_adata.var_names == gene].X.toarray().flatten()
    im0 = axs[0].scatter(x, y, c=color_gt, s=2, alpha=0.9)
    axs[0].set_title("ground truth")
    fig.colorbar(im0)
    
    axs[1].imshow(img_raw)
    color_pred = pred_avg[:, gene_idx]
    im1 = axs[1].scatter(x, y, c=color_pred, s=2, alpha=0.9) 
    axs[1].set_title("pred")
    fig.colorbar(im1)
    
    plt.show()